# Trabalho de Ciência de Dados - Exercício 1

**Aluno:** Iniciante em Ciência de Dados  
**Objetivo:** Exploração de dados, classificação de atributos e aplicação de técnicas de anonimização.

## 1. Exploração e Tipologia

Primeiro, vamos carregar o dataset de previsão de insuficiência cardíaca e classificar os atributos.

In [1]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import hashlib

# Carregando o dataset
file_path = "heart.csv"
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "fedesoriano/heart-failure-prediction",
    file_path,
)

print("Primeiras 5 linhas do dataset:")
display(df.head())

/tmp/ipykernel_10084/2539128710.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'heart-failure-prediction' dataset.
Primeiras 5 linhas do dataset:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


### Tabela de Tipologia dos Atributos

Abaixo está a classificação dos 12 atributos do dataset:

| Atributo | Classificação |
| :--- | :--- |
| Age | Racional |
| Sex | Nominal |
| ChestPainType | Nominal |
| RestingBP | Racional |
| Cholesterol | Racional |
| FastingBS | Nominal |
| RestingECG | Nominal |
| MaxHR | Racional |
| ExerciseAngina | Nominal |
| Oldpeak | Racional |
| ST_Slope | Ordinal |
| HeartDisease | Nominal |

## 2. Identificação de Identificadores

Para a anonimização, precisamos identificar o que pode expor o paciente:

*   **Identificadores Diretos:** No dataset original não existem (como nome ou CPF), mas vamos criar um `PatientID` que será um identificador direto.
*   **Identificadores Indiretos (Quase-identificadores):** Atributos que combinados podem identificar alguém, como `Age`, `Sex`, `RestingBP` e `Cholesterol`.

## 3. Anonimização de Dados

Nesta etapa, vamos aplicar técnicas para proteger a privacidade dos dados.

In [2]:
# 1. Criar a coluna de ID do paciente
df['PatientID'] = [f'PATIENT_{i:04d}' for i in range(len(df))]

# 2. Pseudo-anonimização (Tabela de correspondência)
tabela_correspondencia = df[['PatientID']].copy()
tabela_correspondencia['PseudoID'] = [f'ID_{i+1000}' for i in range(len(df))]

# Aplicando o PseudoID no dataframe principal
df = df.merge(tabela_correspondencia, on='PatientID')
df_anonimizado = df.drop(columns=['PatientID'])

# 3. Código Hash para o PseudoID
def gerar_hash(texto):
    return hashlib.sha256(texto.encode()).hexdigest()[:10]

df_anonimizado['HashID'] = df_anonimizado['PseudoID'].apply(gerar_hash)

# 4. Anonimizar Age (cálculo linear: (idade * 1.5) + 10)
df_anonimizado['Age'] = df_anonimizado['Age'].apply(lambda x: (x * 1.5) + 10)

# 5. Anonimizar Sex (troca de rótulos: M -> 1, F -> 0)
df_anonimizado['Sex'] = df_anonimizado['Sex'].map({'M': 1, 'F': 0})

# 6. Anonimizar ST_Slope (codificação posicional: Up -> 1, Flat -> 2, Down -> 3)
slope_map = {'Up': 1, 'Flat': 2, 'Down': 3}
df_anonimizado['ST_Slope'] = df_anonimizado['ST_Slope'].map(slope_map)

print("Tabela de Correspondência (primeiras linhas):")
display(tabela_correspondencia.head())

print("\nDataset Anonimizado (primeiras linhas):")
display(df_anonimizado.head())

Tabela de Correspondência (primeiras linhas):


,PatientID,PseudoID
0,PATIENT_0000,ID_1000
1,PATIENT_0001,ID_1001
2,PATIENT_0002,ID_1002
3,PATIENT_0003,ID_1003
4,PATIENT_0004,ID_1004



Dataset Anonimizado (primeiras linhas):


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease,PseudoID,HashID
0,70.0,1,ATA,140,289,0,Normal,172,N,0.0,1,0,ID_1000,17824056c4
1,83.5,0,NAP,160,180,0,Normal,156,N,1.0,2,1,ID_1001,cf1ddd4f6c
2,65.5,1,ATA,130,283,0,ST,98,N,0.0,1,0,ID_1002,4df34e3b3f
3,82.0,0,ASY,138,214,0,Normal,108,Y,1.5,2,1,ID_1003,cfa597c294
4,91.0,1,NAP,150,195,0,Normal,122,N,0.0,1,0,ID_1004,299fcf2a48
